In [20]:
# Importing all the required libraries
import os
import json
import re
from pathlib import Path
from tqdm.auto import tqdm

# Mount google drive
from google.colab import drive
drive.mount('/content/drive')

# Setting up the folder paths
PROJECT_ROOT = Path("/content/drive/MyDrive/AI_Loan_Advisory_Chatbot")
PROCESSED_DIR = PROJECT_ROOT / "processed"
CLEANED_TEXT_DIR = PROCESSED_DIR / "cleaned_text"
METADATA_DIR = PROCESSED_DIR / "metadata"
CHUNKS_DIR = PROCESSED_DIR / "chunks"

# Create the chunks folder if it does not exist
if not os.path.exists(CHUNKS_DIR):
    os.makedirs(CHUNKS_DIR)

def detect_document_type(text):
    """Identify the overall structure of a document."""
    # Keywords to check if it's a heading-based document
    SECTION_KEYWORDS = ["eligibility","interest rate","documents required", "processing fee",
                        "loan amount","repayment","security","moratorium", "prepayment",
                        "foreclosure","features","benefits","terms and conditions"]

    # Convert to lower case for easy matching
    text_lower = text.lower()

    # Check if it is a FAQ document
    q_matches = re.findall(r"(?im)^q\.?|^question", text)
    if len(q_matches) >= 2:
        return "FAQ"

    # Check for clause document (e.g. 1.1, 1.2)
    clause_matches = re.findall(r"(?m)^\s*\d+(?:\.\d+)*\.", text)
    if len(clause_matches) >= 8:
        return "CLAUSE"

    # Count how many section keywords are present
    match_count = 0
    for keyword in SECTION_KEYWORDS:
        if keyword in text_lower:
            match_count += 1

    if match_count >= 3:
        return "HEADING"

    # If nothing matches, return plain
    return "PLAIN"

def load_processed_documents(cleaned_text_dir, metadata_dir):
    """Load the metadata and text files saved from Notebook 1."""
    # Empty list to store all documents
    documents = []

    # Find all json files
    metadata_files = list(metadata_dir.rglob("*.json"))

    # Loop through each file using tqdm for a nice progress bar
    for metadata_path in tqdm(metadata_files, desc="Loading Documents"):
        # Open and read metadata
        with open(metadata_path, "r", encoding="utf-8") as f:
            metadata = json.load(f)

        # Get the path for the text file
        relative_path = metadata_path.relative_to(metadata_dir)
        text_path = cleaned_text_dir / relative_path.parent / f"{metadata_path.stem}.txt"

        # Check if text file exists before reading
        if text_path.exists():
            with open(text_path, "r", encoding="utf-8") as f:
                cleaned_text = f.read()

            # Create a dictionary for the document and add it to the list
            doc_dict = {}
            doc_dict["metadata"] = metadata
            doc_dict["analysis"] = {}
            doc_dict["analysis"]["document_structure"] = detect_document_type(cleaned_text)
            doc_dict["content"] = {}
            doc_dict["content"]["cleaned_text"] = cleaned_text

            documents.append(doc_dict)

    return documents

# Calling the function to load everything
documents = load_processed_documents(CLEANED_TEXT_DIR, METADATA_DIR)
print("\n✅ Successfully loaded", len(documents), "documents!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Loading Documents:   0%|          | 0/36 [00:00<?, ?it/s]


✅ Successfully loaded 36 documents!


In [21]:
# Setting word limits for chunks
MIN_WORDS = 20
MAX_WORDS = 250

# Helper function to create a standardized chunk dictionary
def create_chunk(section, text):
    chunk_dict = {
        "section": section.strip(),
        "text": text.strip()
    }
    return chunk_dict

def chunk_faq(text):
    # Split by Q. or Question
    parts = re.split(r'(?=Q\.|Question)', text)
    chunks = []

    for part in parts:
        part = part.strip()
        if part != "":
            # Get the first line to use as the section name
            first_line = part.split("\n")[0]
            chunks.append(create_chunk(first_line, part))

    return chunks

def chunk_clauses(text):
    # Split by clause numbers using regex
    parts = re.split(r'(?=\n\s*\d+(?:\.\d+)*)', text)
    chunks = []
    clause_no = 1

    for part in parts:
        if part != None and part.strip() != "":
            chunks.append(create_chunk(f"Clause {clause_no}", part.strip()))
            clause_no = clause_no + 1

    return chunks

def chunk_headings(text):
    # Regex pattern for common banking headings
    pattern = r'(Eligibility|Interest Rate[s]?|Documents Required|Processing Fee[s]?|Repayment|Moratorium|Prepayment|Foreclosure|Security|Benefits|Features)'
    matches = list(re.finditer(pattern, text, flags=re.IGNORECASE))

    # If no headings found, return as General
    if len(matches) == 0:
        return [create_chunk("General", text)]

    chunks = []
    for i in range(len(matches)):
        start = matches[i].start()

        # Find the end position for slicing
        if i + 1 < len(matches):
            end = matches[i + 1].start()
        else:
            end = len(text)

        section_name = matches[i].group()
        chunk_text = text[start:end]

        chunks.append(create_chunk(section_name, chunk_text))

    return chunks

def chunk_plain_text(text):
    # Split by double newline (paragraphs)
    paragraphs = text.split("\n\n")
    chunks = []
    current_chunk = ""

    for paragraph in paragraphs:
        # Check the length before adding the next paragraph
        temp_text = current_chunk + "\n\n" + paragraph
        if len(temp_text.split()) <= MAX_WORDS:
            current_chunk = temp_text
        else:
            chunks.append(create_chunk("General", current_chunk))
            current_chunk = paragraph

    # Add the last remaining chunk if it's not empty
    if current_chunk != "":
        chunks.append(create_chunk("General", current_chunk))

    return chunks

def enforce_size_limits(chunks, max_words=250):
    # Make sure no chunk is larger than MAX_WORDS
    final_chunks = []

    for chunk in chunks:
        word_count = len(chunk["text"].split())

        if word_count <= max_words:
            final_chunks.append(chunk)
        else:
            # If the chunk is too big, break it down using the plain text method
            sub_chunks = chunk_plain_text(chunk["text"])
            for sub in sub_chunks:
                final_chunks.append(create_chunk(chunk["section"], sub["text"]))

    return final_chunks

def generate_chunks(document):
    """Route the document to the correct chunking strategy."""
    text = document["content"]["cleaned_text"]
    structure = document["analysis"]["document_structure"]

    # Call the right function based on document structure
    if structure == "FAQ":
        raw_chunks = chunk_faq(text)
    elif structure == "CLAUSE":
        raw_chunks = chunk_clauses(text)
    elif structure == "HEADING":
        raw_chunks = chunk_headings(text)
    else:
        raw_chunks = chunk_plain_text(text)

    # Enforce size limits and return the final chunks
    final_chunks = enforce_size_limits(raw_chunks, MAX_WORDS)
    return final_chunks

In [22]:
chunk_counter = 1
all_chunks = []

def build_chunk(document, chunk):
    global chunk_counter
    metadata = document["metadata"]

    # Using .get() just in case a metadata field is missing
    chunk_object = {
        "chunk_id": chunk_counter,
        "bank": metadata.get("bank", "Unknown"),
        "loan_type": metadata.get("loan_type", "Unknown"),
        "document_type": metadata.get("document_type", "Unknown"),
        "section": chunk["section"],
        "source_file": metadata.get("file_name", "Unknown"),
        "page_number": metadata.get("page_number", "N/A"),
        "chunk_text": chunk["text"],
        "word_count": len(chunk["text"].split())
    }

    chunk_counter += 1
    return chunk_object

# Loop through all documents and generate chunks
print("Generating chunks...")
for doc in documents:
    doc_chunks = generate_chunks(doc)

    for chunk in doc_chunks:
        # Check minimum word count to filter out noisy chunks
        if len(chunk["text"].split()) >= MIN_WORDS:
            final_chunk = build_chunk(doc, chunk)
            all_chunks.append(final_chunk)

# Grouping chunks by bank and loan type before saving
grouped_chunks = {}

for chunk in all_chunks:
    bank = chunk["bank"]
    loan = chunk["loan_type"]

    # If bank is not in dictionary, add it
    if bank not in grouped_chunks:
        grouped_chunks[bank] = {}

    # If loan type is not in bank dictionary, add it
    if loan not in grouped_chunks[bank]:
        grouped_chunks[bank][loan] = []

    grouped_chunks[bank][loan].append(chunk)

# Saving the chunks to folders
total_files_saved = 0

for bank in grouped_chunks:
    for loan in grouped_chunks[bank]:
        chunks_to_save = grouped_chunks[bank][loan]

        # Creating save directory
        save_dir = CHUNKS_DIR / bank / loan
        if not os.path.exists(save_dir):
            os.makedirs(save_dir)

        # Writing data to a json file
        file_path = save_dir / "chunks.json"
        with open(file_path, "w", encoding="utf-8") as f:
            json.dump(chunks_to_save, f, indent=4, ensure_ascii=False)

        total_files_saved += 1

print("============================================================")
print("Total Chunks Generated :", len(all_chunks))
print("Saved into", total_files_saved, "JSON files inside /processed/chunks/")
print("============================================================")

Generating chunks...
Total Chunks Generated : 241
Saved into 9 JSON files inside /processed/chunks/
